#Approch
Load movie dataset
Extract genres
Convert genres → text features
Compute TF-IDF vectors
Compute cosine similarity between movies
Create recommendation function

In [3]:
pip install scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 9.6 MB/s  0:00:009.7 MB/s eta 0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 13.9 MB/s  0:00:02 14.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn]0m 3/4 [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Import PySpark session to work with big data
from pyspark.sql import SparkSession  # used to start spark environment

# Import pandas for local dataframe operations
import pandas as pd  # used for easier processing for TF-IDF

# Import TF-IDF vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer  # converts text into TF-IDF vectors

# Import cosine similarity
from sklearn.metrics.pairwise import cosine_similarity  # computes similarity between vectors

In [6]:
spark = SparkSession.builder.appName("MovieLensRecommender").getOrCreate()  # start spark session


26/03/14 17:46:13 WARN Utils: Your hostname, rajesh-pc resolves to a loopback address: 127.0.1.1; using 192.168.0.39 instead (on interface wlp1s0)
26/03/14 17:46:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/14 17:46:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
# Path to your dataset
data_path = "/home/rajesh/CSL7100/Assignment3/ml-latest-small/"

# Load movies dataset using Spark
movies_spark = spark.read.csv(data_path + "movies.csv", header=True, inferSchema=True)  
# header=True reads column names, inferSchema automatically detects column types

movies_spark.show(5)  # preview dataset

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows



In [8]:
movies = movies_spark.toPandas()  
# convert Spark dataframe to pandas because sklearn works with pandas/numpy

In [27]:
movies.shape

(9742, 3)

In [9]:
movies["genres"] = movies["genres"].str.replace("|", " ")  
# replace "|" with space so TF-IDF treats each genre as a word

In [13]:
print(movies["genres"].head())

0    Adventure Animation Children Comedy Fantasy
1                     Adventure Children Fantasy
2                                 Comedy Romance
3                           Comedy Drama Romance
4                                         Comedy
Name: genres, dtype: str


In [14]:
tfidf = TfidfVectorizer(stop_words="english")  
# create TF-IDF vectorizer; removes common words if present

tfidf_matrix = tfidf.fit_transform(movies["genres"])  
# convert movie genres into TF-IDF feature vectors

In [28]:
print(tfidf_matrix.shape)

(9742, 23)


In [17]:
print(tfidf_matrix[:5])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 14 stored elements and shape (5, 23)>
  Coords	Values
  (0, 1)	0.41684567364693936
  (0, 2)	0.5162254711770092
  (0, 3)	0.5048454681396087
  (0, 4)	0.26758647689140014
  (0, 8)	0.482990142708577
  (1, 1)	0.5123612074824269
  (1, 3)	0.6205251727456431
  (1, 8)	0.5936619434123594
  (2, 4)	0.5709154064399099
  (2, 18)	0.8210088907493954
  (3, 4)	0.5050154397005037
  (3, 18)	0.726240982959826
  (3, 7)	0.46640480307738325
  (4, 4)	1.0


In [18]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)  
# compute similarity between every pair of movies

In [29]:
print(cosine_sim.shape)

(9742, 9742)


In [19]:
print(cosine_sim[:5])

[[1.         0.81357774 0.15276924 ... 0.         0.4210373  0.26758648]
 [0.81357774 1.         0.         ... 0.         0.         0.        ]
 [0.15276924 0.         1.         ... 0.         0.         0.57091541]
 [0.1351353  0.         0.8845714  ... 0.4664048  0.         0.50501544]
 [0.26758648 0.         0.57091541 ... 0.         0.         1.        ]]


In [20]:
indices = pd.Series(movies.index, index=movies["title"]).drop_duplicates()  
# create mapping: movie title -> dataframe index

In [30]:
print(indices.shape)

(9742,)


In [21]:
def recommend_movies(title, top_n=5):
    
    idx = indices[title]  
    # get index of the movie given by user
    
    sim_scores = list(enumerate(cosine_sim[idx]))  
    # get similarity scores of this movie with all movies
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)  
    # sort movies based on similarity score
    
    sim_scores = sim_scores[1:top_n+1]  
    # skip the first one (it is the movie itself)
    
    movie_indices = [i[0] for i in sim_scores]  
    # extract indices of most similar movies
    
    results = movies.iloc[movie_indices][["title"]]  
    # get movie titles
    
    results["similarity_score"] = [i[1] for i in sim_scores]  
    # add similarity score
    
    return results

In [22]:
recommend_movies("Toy Story (1995)", top_n=5)

,title,similarity_score
1706,Antz (1998),1.0
2355,Toy Story 2 (1999),1.0
2809,"Adventures of Rocky and Bullwinkle, The (2000)",1.0
3000,"Emperor's New Groove, The (2000)",1.0
3568,"Monsters, Inc. (2001)",1.0


In [26]:
recommend_movies("Tom and Huck (1995)", top_n=5)

,title,similarity_score
119,"Amazing Panda Adventure, The (1995)",1.0
131,Casper (1995),1.0
204,Far From Home: The Adventures of Yellow Dog (1...,1.0
421,Lassie (1994),1.0
521,Homeward Bound II: Lost in San Francisco (1996),1.0


In [31]:
spark.stop()